# 3D-SynTree: Execution & Training Engine
**Structure-Based Molecular Design via Reaction-Constrained Synthon Assembly**

**Pipeline:** Setup → Clone → Dependencies → Asset Verification → Prebuilt Dataset → GPU → Train → Status\ndata processing, training, and checkpointing logic resides strictly within the codebase.

**Pipeline:** Setup → Clone → Dependencies → Assets → GPU Detect → Run Codebase → Status

**Hugging Face:** you will be **asked for your HF write token** in Cell 1 (a hidden input
prompt). Leave it blank to keep checkpoints local-only. Tokens are also auto-detected from
Colab/Kaggle secrets or the `HF_TOKEN` environment variable if present. The token is
validated immediately (account + write permission), so a read-only or wrong token is caught
with a clear message *before* training starts instead of failing with a 403 mid-run.

**GPU auto-scaling:** the training cell grows the model (up to hidden 256 / 8 layers) and
auto-tunes the batch size so the GPU is filled to ~85% of its free VRAM (≈12 GB on a T4),
The notebook consumes the prebuilt, reaction-validated Hugging Face dataset; synthetic data is restricted to explicit smoke-test runs.\nthe first build), which at batch ~640 gives ~100 optimizer steps per epoch.


In [ ]:
# CELL 1: Environment & Configuration
import os, json

# 1. Hugging Face token: env var / platform secrets first, then ASK directly.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            HF_TOKEN = ""

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("\nEnter your HuggingFace WRITE token (blank = skip Hub sync): ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
print(f"HF token {'active - checkpoints will sync to the Hub.' if HF_TOKEN else 'NOT set - checkpoints stay local only.'}")

# 1b. Validate the token NOW (account + write permission) so a bad token fails
# here with a clear message instead of a 403 halfway through training.
if HF_TOKEN:
    try:
        from huggingface_hub import whoami
        info = whoami(token=HF_TOKEN)
        user = info.get("name", "?")
        role = (info.get("auth", {}).get("accessToken", {}) or {}).get("role", "unknown")
        if role in ("write", "contributor", "admin"):
            print(f"HF token validated: account '{user}', access role '{role}' - Hub sync ready.")
        else:
            print(f"WARNING: token for account '{user}' has role '{role}' - it CANNOT create or push repos.")
            print("Fix: create a token with WRITE access at https://huggingface.co/settings/tokens")
            print("     (fine-grained: 'Read and write' on repositories; classic token: 'write' scope).")
    except Exception as e:
        print(f"WARNING: could not validate the HF token ({type(e).__name__}: {e}).")
        print("Training still runs; checkpoints just stay local if the token is unusable.")

# 2. Runtime execution configuration - EDIT THESE to your account.
RUNTIME_CONFIG = {
    "repo_url": "https://github.com/Vtheonly/3d-syntree.git",
    "branch": "main",
    "hf_repo_id": "Vtheonly/3d-syntree-checkpoints",
    "config_override": {
        "huggingface": {
            "enabled": bool(HF_TOKEN),
            "repo_id": "Vtheonly/3d-syntree-checkpoints",
            "push_every_n_epochs": 2
        },
        "training": {
            "time_budget_hours": 11.5,
            "auto_scale": {
                "enabled": True,
                "target_vram_fraction": 0.85,
                "max_batch_size": 8192,
                "scale_model": True
            }
        },
        "data": {
            "synthetic_samples": 65536
        }
    }
}

with open("runtime_config.json", "w") as f:
    json.dump(RUNTIME_CONFIG, f, indent=2)
print("Configuration profile initialized (GPU auto-scaling: ON, target 85% of free VRAM).")

In [ ]:
# CELL 2: Clone or Pull Repository
import os

REPO_DIR = "3d-syntree"
REPO_URL = RUNTIME_CONFIG["repo_url"]
BRANCH = RUNTIME_CONFIG["branch"]

if os.path.exists(REPO_DIR):
    print(f"Pulling latest changes in {REPO_DIR}...")
    !cd {REPO_DIR} && git checkout {BRANCH} && git pull
else:
    print(f"Cloning {REPO_URL}...")
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("Repository synchronization complete.")

In [ ]:
# CELL 4: Verify the Exact Synthon Catalog Used by the Dataset\nCATALOG_PATH = os.environ.get(\"SYNTHREE_CATALOG_PATH\", \"./data/enamine_3d_subset.parquet\")\nif not os.path.exists(CATALOG_PATH):\n    raise FileNotFoundError(\n        f\"Missing {CATALOG_PATH}. The prebuilt dataset's target indices are tied to the exact catalog used during preprocessing; do not silently substitute the synthetic smoke catalog.\"\n    )\nprint(f\"Verified synthon catalog: {CATALOG_PATH}\")\n

In [ ]:
# CELL 5: Bind to the Prebuilt Hugging Face Dataset\nDATASET_REPO = os.environ.get(\"SYNTHREE_DATASET_REPO\", \"Vtheonly/cleaned-sbdd-multidataset\")\nDATASET_REVISION = \"main\"\nRUNTIME_CONFIG[\"config_override\"].setdefault(\"data\", {})\nRUNTIME_CONFIG[\"config_override\"][\"data\"].update({\n    \"backend\": \"huggingface\",\n    \"huggingface\": {\"repo_id\": DATASET_REPO, \"revision\": DATASET_REVISION, \"cache_dir\": \"./hf_cache\", \"max_cached_shards\": 2}\n})\nwith open(\"../runtime_config.json\", \"w\") as f:\n    json.dump(RUNTIME_CONFIG, f, indent=2)\nfrom huggingface_hub import hf_hub_download\nfor _split in (\"train\", \"val\", \"test\"):\n    manifest_path = hf_hub_download(repo_id=DATASET_REPO, filename=f\"data/{_split}/manifest.json\", repo_type=\"dataset\", revision=DATASET_REVISION, token=HF_TOKEN or None)\n    with open(manifest_path) as f:\n        manifest = json.load(f)\n    print(f\"{_split}: {manifest['total_samples']} samples across {len(manifest['shards'])} shards\")\n

In [ ]:
# CELL 5: Download External Assets (Data & Synthon Catalog)
print("Executing headless data and asset retrieval script...")
!python scripts/download_assets.py \
    --target-dataset crossdocked2020 \
    --synthon-subset 3d-diversity-15k \
    --output-dir ./data

print("External assets and synthon libraries ready.")

In [ ]:
# CELL 6: Hardware & Device Verification
import json
import torch
from syntree.utils.hardware import configure_runtime_environment, free_vram_bytes

device_info = configure_runtime_environment()
print("Hardware execution profile:")
print(json.dumps(device_info, indent=2))

if torch.cuda.is_available():
    free_gb = free_vram_bytes() / 1024**3
    print(f"Free VRAM: {free_gb:.2f} GB -> auto-scale target: {0.85 * free_gb:.2f} GB")

assert device_info["device"].startswith("cuda"), \
    "No GPU detected: enable GPU acceleration in Runtime > Change runtime type."

In [ ]:
# CELL 7: Execute Codebase Training / Evaluation Pipeline
# GPU auto-scaling is enabled: expect [main] auto-scale / [trainer] auto-scale
# lines growing the model and batch size to ~85% of free VRAM before epoch 0.
print("Launching 3D-SynTree training engine via CLI entrypoint...")

!python main.py \
    --mode train \
    --config configs/train_colab_12h.json \
    --runtime-config ../runtime_config.json \
    --resume-auto

In [ ]:
# CELL 8: Final Status, Benchmark Artifacts & HF Hub Verification
import json, os
from syntree.utils.checkpoint import verify_hf_sync

status = verify_hf_sync(RUNTIME_CONFIG["hf_repo_id"])
print("=== EXECUTION RUN COMPLETE ===")
print(f"Latest Checkpoint: {status['latest_remote_checkpoint']}")
print(f"Total Epochs Completed: {status['epochs_completed']}")
print(f"Hugging Face Sync Verified: {status['sync_ok']}")
if "error" in status:
    print(f"(sync note: {status['error']})")

if os.path.exists("experiments/latest_metrics.json"):
    with open("experiments/latest_metrics.json") as f:
        metrics = json.load(f)
    print("\nValidation Summary:")
    print(json.dumps(metrics, indent=2))

# Optional: generate ligands + recipes for the sample pocket and benchmark them.
# !python main.py --mode generate --num-ligands 8 --resume-auto
# !python main.py --mode evaluate --sdf-dir ./outputs